In [1]:
import pandas as pd
import numpy as np 
import math
import matplotlib.pyplot as plt
import joblib
import os 

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error




from scipy.sparse.linalg import svds
from flask import Flask, request, jsonify
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


from surprise import Dataset, Reader , accuracy
from surprise.prediction_algorithms.matrix_factorization import SVD
from surprise.model_selection import GridSearchCV


from sklearn.model_selection import train_test_split 
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
moviesDF = pd.read_csv("./ml-1m/movies.dat" , delimiter="::", encoding="latin-1" ,names=["MovieID", "Title", "Genres"])
ratingsDF = pd.read_csv("./ml-1m/ratings.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "MovieID", "Rating", "Timestamp"])
usersDF = pd.read_csv("./ml-1m/users.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "Gender", "Age", "Occupation", "Zip-code"])

C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_31020\336456026.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  moviesDF = pd.read_csv("./ml-1m/movies.dat" , delimiter="::", encoding="latin-1" ,names=["MovieID", "Title", "Genres"])
C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_31020\336456026.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratingsDF = pd.read_csv("./ml-1m/ratings.dat" , delimiter="::", encoding="latin-1" ,names=["UserID", "MovieID", "Rating", "Timestamp"])
C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_31020\336456026.py:3: ParserWarning: Falling back to the 'python' en

In [3]:
ratingsDFPivoted = ratingsDF.pivot(index="UserID", columns="MovieID", values="Rating").fillna(0)
ratingsDFPivoted

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
print("movies cols:", moviesDF.columns) 
print("ratings cols:", ratingsDF.columns)
print("user DF:", usersDF.columns)

print(moviesDF.shape)
print(ratingsDF.shape)
print(usersDF.shape)

movies cols: Index(['MovieID', 'Title', 'Genres'], dtype='object')
ratings cols: Index(['UserID', 'MovieID', 'Rating', 'Timestamp'], dtype='object')
user DF: Index(['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'], dtype='object')
(3883, 3)
(1000209, 4)
(6040, 5)


<h3><b>Separate the non-rated and rated movies </b></h3>

In [4]:
rated_movies = ratingsDF.merge(moviesDF, on="MovieID")

rated_movies = moviesDF.loc[moviesDF["MovieID"].isin(rated_movies["MovieID"].unique())] 
print("rated movies shape: ",rated_movies.shape)

non_rated_movies = moviesDF.loc[~moviesDF["MovieID"].isin(rated_movies["MovieID"].unique())] 
print("Non rated movies shape: ", non_rated_movies.shape)

rated_movies["Title"] = rated_movies["Title"].str.lower()

non_rated_movies.head()

rated movies shape:  (3706, 3)
Non rated movies shape:  (177, 3)


C:\Users\niaz mahmud\AppData\Local\Temp\ipykernel_31020\1260981810.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rated_movies["Title"] = rated_movies["Title"].str.lower()


,MovieID,Title,Genres
50,51,Guardian Angel (1994),Action|Drama|Thriller
107,109,Headless Body in Topless Bar (1995),Comedy
113,115,Happiness Is in the Field (1995),Comedy
141,143,Gospa (1995),Drama
281,284,New York Cop (1996),Action|Crime


<h3><b>Check the sparsity: how many movies not yet rated by all user in percentage</b></h3>

In [6]:
n_unique_user = ratingsDF["UserID"].unique().shape
n_unique_movies = ratingsDF["MovieID"].unique().shape

print(n_unique_user, n_unique_movies)

spaecity = round(1.0 - len(ratingsDF)/float(n_unique_user[0]*n_unique_movies[0]) , 3)# round in 3 decimal point
print(f"The Sparcity level is : {spaecity*100}%")

(6040,) (3706,)
The Sparcity level is : 95.5%


<h2><b>Now apply SVD from Surprise</b></h2>

In [7]:
svd_merged_movie_rating = ratingsDF.merge(moviesDF, on="MovieID", how="left")
# svd_merged_movie_rating["Rating"] = svd_merged_movie_rating["Rating"]/max(svd_merged_movie_rating["Rating"])

print(svd_merged_movie_rating.shape)
print(max(svd_merged_movie_rating["UserID"]))
svd_merged_movie_rating.head()

(1000209, 6)
6040


,UserID,MovieID,Rating,Timestamp,Title,Genres
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy


In [8]:
def splitter():
    svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df = train_test_split(svd_merged_movie_rating, test_size = 0.001)
    return (svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df)
  
while True: 
    svd_merged_movie_rating_train_df, svd_merged_movie_rating_test_df = splitter()
    if sorted(svd_merged_movie_rating_train_df["UserID"].unique()) == [x for x in range(1, 6041)]: 
        break

In [9]:
# create and apply reader in train dataaset
reader = Reader(
    rating_scale=(
        min(svd_merged_movie_rating["Rating"].to_numpy()),
        max(svd_merged_movie_rating["Rating"].to_numpy())
    )
)

svd_reader_train = Dataset.load_from_df(svd_merged_movie_rating_train_df[['UserID', 'MovieID', 'Rating']], reader)
svd_reader_trainset = svd_reader_train.build_full_trainset() # this will use while training with "SVD" not for Hyper-parameter-tuner

<h4><b>Apply hyper parameter tuning on SVD usnig "GridSearchCV"</b></h4>

In [10]:
hyperParamGrids = {
    "n_factors": [50, 100, 150, 200],
    "lr_all" : [0.002, 0.003, 0.005, 0.007, 0.009, 0.01, 0.02, 0.05],
    "reg_all": [0.002, 0.003, 0.005, 0.007, 0.009, 0.01, 0.02, 0.05]
}

gridSearch = GridSearchCV(
    algo_class = SVD, 
    param_grid = hyperParamGrids, 
    measures=["rmse", "mae"], 
    cv= 3, 
    refit=False , 
    joblib_verbose = 2,
    n_jobs=-1, 
)

gridSearch.fit(svd_reader_train)


print("Best RMSE score:", gridSearch.best_score['rmse'])
print("Best MAE score:", gridSearch.best_score['mae'])

print()
print("Best hyperparameters in RMSE:")
print("n_factors:", gridSearch.best_params['rmse']['n_factors'])
print("lr_all:", gridSearch.best_params['rmse']['lr_all'])
print("reg_all:", gridSearch.best_params['rmse']['reg_all'])


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


KeyboardInterrupt: 

In [ ]:
model = SVD(
    n_factors= gridSearch.best_params['rmse']['n_factors'], 
    n_epochs= 200,
    biased= True,
    lr_all = gridSearch.best_params['rmse']['lr_all'],
    reg_all= gridSearch.best_params['rmse']['reg_all'], #
    verbose = True
)
model.fit(svd_reader_trainset)

In [ ]:
prediction = model.predict(uid=5050, iid=51)
print(prediction.est)

In [ ]:
# now validate the model 

testlist = list(zip(svd_merged_movie_rating_test_df["UserID"], svd_merged_movie_rating_test_df["MovieID"], svd_merged_movie_rating_test_df["Rating"])) 
# it is list of tuples: [(UserID, MovieID, Rating), (UserID, MovieID, Rating),(UserID, MovieID, Rating).......]


pred2 = model.test(testlist)
print(accuracy.mae(pred2))
print(accuracy.mse(pred2))
print(accuracy.rmse(pred2))

<h5><b> Now get the pu-> user latent vector and qi-> item latent vector</b></h5>

In [ ]:
user_factors = model.pu 
item_factors = model.qi

print(f"User latent vector: {user_factors.shape}")
print(f"Item latent vector: {item_factors.shape}")

<h7><b>as SVD pu and qi organized the user and differnt order so I need organize them manually</b></h7>

In [ ]:
svd_userBias = []
svd_itemBias = []

user_vector = []
item_vector = []

# fill svd_userBias and user_vector
for x in sorted(svd_merged_movie_rating["UserID"].unique(), reverse=False): 
    user_inner_id = svd_reader_trainset.to_inner_uid(x)
    
    svd_userBias.append(model.bu[user_inner_id])
    user_vector.append(model.pu[user_inner_id])
    
    
# fill svd_itemBias and item_vector
for x in sorted(svd_merged_movie_rating["MovieID"].unique(), reverse=False): 
    movie_inner_id = svd_reader_trainset.to_inner_iid(x)
    
    svd_itemBias.append(model.bi[movie_inner_id])
    item_vector.append(model.qi[movie_inner_id])
    
    
    
svd_userBias = np.array(svd_userBias)
svd_itemBias = np.array(svd_itemBias)

user_vector = np.array(user_vector)
item_vector = np.array(item_vector)

In [ ]:
SVD_Final_Vector_table = user_vector @ item_vector.T + svd_userBias.reshape(-1, 1) + svd_itemBias.reshape(1, -1) + svd_reader_trainset.global_mean

SVD_rated_Vector_table = pd.DataFrame(SVD_Final_Vector_table , columns=sorted(svd_merged_movie_rating["MovieID"].unique(), reverse=False)) 

SVD_non_rated_Vector_table = np.full((6040, len(non_rated_movies["MovieID"].unique())), svd_reader_trainset.global_mean)
SVD_non_rated_Vector_table = pd.DataFrame(SVD_non_rated_Vector_table, columns=non_rated_movies["MovieID"].unique())

SVD_Final_Vector_table = pd.concat([SVD_rated_Vector_table, SVD_non_rated_Vector_table], axis=1)
SVD_Final_Vector_table = SVD_Final_Vector_table[sorted(SVD_Final_Vector_table.columns)]

print(SVD_Final_Vector_table.shape)
SVD_Final_Vector_table

In [ ]:
table = svd_merged_movie_rating_test_df.copy()

actual_rating = []
predicted_rating = []


for uid in sorted(table["UserID"].unique()): 
    userData = table.loc[table["UserID"] == uid].sort_values(by="MovieID", ascending=True)
    user_MovieId = userData["MovieID"]
    
    user_Actaual_Rating = list(userData["Rating"].to_numpy())    
    
    model_pred = []
    for mid in user_MovieId: 
        model_pred.append(SVD_Final_Vector_table.loc[uid-1][mid])
    
    actual_rating.extend(user_Actaual_Rating)
    predicted_rating.extend(model_pred)
    
print(len(actual_rating))
print(len(predicted_rating))
    

In [ ]:
mseii = mean_squared_error(actual_rating, predicted_rating)
rmseii = np.sqrt(mseii)
print(f"mse:{mseii/5.00:0.3f}  ||  RMSE:{rmseii/5.00:0.3f}")

<h2><b> Start work with Content(TF-IDF) filtering ========</b></h2>

<h5>Now build TF-IDF tables for rated and non-rated movies: =====================</h5>

In [5]:
prep_all_movies = ColumnTransformer(
    transformers=[
        ('genresTFIDF',  TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(1,18),
            max_df=0.9,
            min_df=1,
            max_features=500,
            analyzer="word", 
            token_pattern=r'[^|]+'
        ), 'Genres')
    ],
    remainder='drop'
)

prep_all_movies.fit(moviesDF)

ColumnTransformer(transformers=[('genresTFIDF',
                                 TfidfVectorizer(max_df=0.9, max_features=500,
                                                 ngram_range=(1, 18),
                                                 stop_words='english',
                                                 token_pattern='[^|]+'),
                                 'Genres')])

<h5>Now build TF-IDF tables for rated movies: ====================== </h5>

* main goal here is: to create a TF-IDF table to expalain - how the rated movies(3706 number of movies) matches to the rated-movies(3706 number of movies)

In [6]:
rated_tfidfTable = prep_all_movies.transform(rated_movies)

rated_cosineSim = cosine_similarity(rated_tfidfTable, rated_tfidfTable) # value range: 0-1
print(rated_cosineSim.shape) # (number of rated movies , number of rated movies)


rated_tfidf_df =pd.DataFrame(rated_cosineSim, columns=rated_movies["MovieID"].to_numpy()) # shape -> (number of user , number of rated movies)
print(rated_tfidf_df.shape) 


(3706, 3706)
(3706, 3706)


In [ ]:

dense_matrix = rated_tfidfTable.toarray()  # or sparse_matrix.A
print(dense_matrix)
print(dense_matrix.shape)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(3706, 398)


: 

<h5>Now build TF-IDF tables for non-rated movies: =========================</h5>

* main goal here is: to create a TF-IDF table to expalain - how the non rated movies(177 movies) matches to the rated-movies(3706 number of movies)

In [ ]:
non_rated_tfidfTable = prep_all_movies.transform(non_rated_movies)
print(non_rated_tfidfTable.shape) 


non_rated_cosineSim = cosine_similarity(non_rated_tfidfTable, rated_tfidfTable).T
print(non_rated_cosineSim.shape) # (number of non-rated movies , number of rated movies)

non_rated_tfidf_df = pd.DataFrame(non_rated_cosineSim, columns=non_rated_movies["MovieID"].to_numpy()) # shape -> (number of user , number of rated movies)
print(non_rated_tfidf_df.shape) 

<h4><b> Now combine rated TF-IDF and non rated tf-idf table =====</b></h4>

In [ ]:
combinedPivoted = MinMaxScaler(feature_range=(0, 5))

combined_tfidf_df = pd.concat([rated_tfidf_df, non_rated_tfidf_df] , axis=1)
combined_tfidf_df = combined_tfidf_df[sorted(combined_tfidf_df.columns)]
combined_tfidf_df_t = combinedPivoted.fit_transform(combined_tfidf_df)
combined_tfidf_df = pd.DataFrame(combined_tfidf_df_t , columns=combined_tfidf_df.columns)

combined_tfidf_df

In [ ]:
ratingsDFPivoted = ratingsDF.pivot(index="UserID", columns="MovieID", values="Rating").fillna(0)
ratingsDFPivoted = (ratingsDFPivoted != 0).astype(int)
print(ratingsDFPivoted.shape, combined_tfidf_df.shape)

comb_svd_tfIdf3 = pd.DataFrame(np.dot(ratingsDFPivoted, combined_tfidf_df), columns=combined_tfidf_df.columns)

print(type(comb_svd_tfIdf3))
print(comb_svd_tfIdf3.shape)
comb_svd_tfIdf3

NameError: name 'combined_tfidf_df' is not defined

<h3><b> Conver the tables to dataframe: =========== </b></h3>

In [ ]:
# SVD_Final_Vector_table conversion
SVD_Final_Vector_temp = SVD_Final_Vector_table.reset_index()
SVD_Final_Vector_df = SVD_Final_Vector_temp.melt(
    id_vars='index', 
    var_name='MovieID',
    value_name='SVD_Rating'
)
SVD_Final_Vector_df = SVD_Final_Vector_df.rename(columns={'index': 'UserID'})
SVD_Final_Vector_df["UserID"] +=1

SVD_Final_Vector_df

In [ ]:
# comb_svd_tfIdf3
comb_svd_tfIdf_temp = comb_svd_tfIdf3.reset_index()
comb_svd_tfIdf_df = comb_svd_tfIdf_temp.melt(
    id_vars='index', 
    var_name='MovieID',
    value_name='TfIdf_Rating'
)
comb_tfIdf_df = comb_svd_tfIdf_df.rename(columns={'index': 'UserID'})
comb_tfIdf_df["UserID"] +=1

comb_tfIdf_df

<h3><b> Now Train Gradient boosting to see the result</b></h3>

In [ ]:
comb_df = comb_tfIdf_df.merge(SVD_Final_Vector_df, on=["UserID", "MovieID"], how="left")
comb_df= comb_df.merge(ratingsDF, on=["UserID", "MovieID"], how="left").drop(["Timestamp"], axis=1)

combinedScale1 = MinMaxScaler(feature_range=(0, 5))
comb_df["TfIdf_Rating"] = combinedScale1.fit_transform(comb_df[["TfIdf_Rating"]])

comb_cleaned_df = comb_df.dropna()
comb_cleaned_df

In [ ]:
gradient_x = comb_cleaned_df.iloc[:, 2:4]
gradient_y = comb_cleaned_df.iloc[:, -1]
gradient_train_x, gradient_test_x, gradient_train_y, gradient_test_y = train_test_split(gradient_x, gradient_y, test_size=0.001)

gradient_model = GradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.01, 
    n_estimators= 700, 
    subsample= 0.90, 
    max_depth=7,  
    random_state=42
)
gradient_model.fit(gradient_train_x, gradient_train_y)

In [ ]:
dummyPred = gradient_model.predict([[0.086431, 4.241818]])
print(dummyPred)

In [ ]:
gradient_pred = gradient_model.predict(gradient_test_x)
gradien_rmse = mean_squared_error(gradient_test_y, gradient_pred)
print(f"gradien model accuracy: {gradien_rmse/5.00:.4f}")

<h3><b> Get top N unrated movies from SVD and Tf-Idf array</b></h3>

In [ ]:
ratingsDF

In [ ]:
def getTopN_Movie(SVDArray, Tf_IdfArray, combined_tfidf_dfs, ratingsDF, UID, movieID, ret_n_movie): 
    rated_mov_by_uid = ratingsDF.loc[ratingsDF["UserID"]==UID]
    unrated_mov_by_uid = ratingsDF.loc[~ratingsDF["MovieID"].isin(rated_mov_by_uid["MovieID"])].drop(["Timestamp"], axis=1)["MovieID"].unique()    
    unrated_mov_by_uid = pd.DataFrame({
        "MovieID": unrated_mov_by_uid
    })
    
    
    svd_user_all_rated_movie = SVDArray.loc[UID-1].to_numpy()
    tfIdf_all_movies = Tf_IdfArray.loc[UID-1].to_numpy()
    svd_user_all_rated_movie= pd.DataFrame({
        "MovieID": [x for x in range(1, len(svd_user_all_rated_movie)+1)], 
        "Rating": svd_user_all_rated_movie
    })
    tfIdf_all_movies = pd.DataFrame({
        "MovieID": [x for x in range(1, len(tfIdf_all_movies)+1)], 
        "Rating": tfIdf_all_movies
    })

    
    pure_tfIdf = combined_tfidf_dfs.loc[movieID].to_numpy()

    pure_tfIdf = pd.DataFrame({
        "MovieID": combined_tfidf_dfs.columns, 
        "Pure_TfIdf_Rating": pure_tfIdf
    })
    print(combined_tfidf_dfs.columns)
    
    merged_all = unrated_mov_by_uid.merge(svd_user_all_rated_movie , on=["MovieID"])
    merged_all = merged_all.merge(tfIdf_all_movies, on=["MovieID"])
    merged_all = merged_all.merge(pure_tfIdf, on=["MovieID"])
    merged_all = merged_all.rename(
        columns={"Rating_x": "SVD_Rating" , "Rating_y":"TfIdf_Rating"}
    )
    
    combinedScale = MinMaxScaler(feature_range=(0, 5))
    merged_all["TfIdf_Rating"] = combinedScale.fit_transform(merged_all[["TfIdf_Rating"]])
    
    if 3706-len(unrated_mov_by_uid)>30: 
        # if user rated less that 30 movies then we mostly relay on Tf-IDF
        merged_all["Combined_Ratinng"] =  (0.2*merged_all["SVD_Rating"]) + (0.6*merged_all["TfIdf_Rating"]) + (0.2*merged_all["Pure_TfIdf_Rating"])
    else: 
        # now user has enough rated movies ... we ralay in mostly SVD
        merged_all["Combined_Ratinng"] =  (0.4*merged_all["SVD_Rating"]) + (0.2*merged_all["TfIdf_Rating"])+ (0.4*merged_all["Pure_TfIdf_Rating"])

        
    merged_all = merged_all.sort_values(["Combined_Ratinng"], ascending=False)
    
    return merged_all.iloc[0:ret_n_movie, :]

    
    
        
    
topN_unrated_movies = getTopN_Movie(SVD_Final_Vector_table, comb_svd_tfIdf3, combined_tfidf_df, ratingsDF, UID=4, movieID=50 , ret_n_movie=50)
topN_unrated_movies


<h3><b>Predict the rating of Top-N movies rating using Meta-Level Model </b></h3>

In [ ]:
def meta_predicton(metaModel, topN_df, ret_n_movie):
    metaModel_pred = metaModel.predict(topN_df[["TfIdf_Rating", "SVD_Rating"]])
    print(type(metaModel_pred))

    topN_df["Meta_Model_Rating"] = metaModel_pred
    topN_df = topN_df.sort_values(["Meta_Model_Rating"], ascending=False)
    return topN_df.iloc[0:ret_n_movie, :]


meta_predicton(gradient_model, topN_unrated_movies, 10)

<h3><b> Save SVD and gradient_model </b></h3>

In [ ]:
joblib.dump(gradient_model, 'gradient_model.pkl')
joblib.dump(model, 'svd_model.pkl')

### save the trainset for svd-model

In [ ]:
joblib.dump(svd_reader_trainset, "svd_reader_trainset.pkl")

In [ ]:
# now load the model and get some prediction 
loaded_svd_model = joblib.load('svd_model.pkl')
loaded_gradient_model = joblib.load('gradient_model.pkl')

In [ ]:
print(loaded_svd_model.pu)
print(loaded_svd_model.qi)
print()
print()

gradDF = pd.DataFrame({
    "TfIdf_Rating": [5.0], 
    "SVD_Rating": [4.392598]	
})
print(loaded_gradient_model.predict(gradDF[["TfIdf_Rating", "SVD_Rating"]]))

===============================================================